# Create Uploadable Cholec80-CSV 5 FPS Dataset

This notebook creates a standalone copied dataset containing only the frames referenced by `data/train.csv`, `data/val.csv`, and `data/test.csv`.

It does not move or delete anything from `data/cholec80_frames`.

## 1. Setup

Run this cell first. The notebook can be launched from the repository root or from the `notebooks/` folder.

In [2]:
import csv
import hashlib
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, total=None, desc=None):
        return iterable if iterable is not None else []


def find_project_root(start):
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / "colenet").exists() and (path / "data_preprocessing").exists():
            return path
    raise FileNotFoundError("Could not find the CHOLEC80-CVS-PUBLIC project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SOURCE_FRAMES_DIR = PROJECT_ROOT / "data" / "cholec80_frames"
OUTPUT_DIR = PROJECT_ROOT / "data" / "cholec80_cvs"

SPLIT_CSVS = {
    "train": PROJECT_ROOT / "data" / "train.csv",
    "valid": PROJECT_ROOT / "data" / "val.csv",
    "test": PROJECT_ROOT / "data" / "test.csv",
}

ANNOTATION_ASSETS = {
    "stats.csv": PROJECT_ROOT / "data" / "stats.csv",
    "folds.json": PROJECT_ROOT / "data" / "folds.json",
    "videos_frames_index.csv": PROJECT_ROOT / "results" / "videos_frames_index.csv",
    "config_snapshot.json": PROJECT_ROOT / "config" / "config.json",
}

LABEL_COLUMNS = ["two_structures_score", "cystic_plate_score", "hc_triangle_score"]
ORIGINAL_VIDEO_FPS = 25
TARGET_FPS = 5
EXPECTED_TOTAL_UNIQUE_FRAMES = 62752
EXPECTED_FULL_EXTRACTED_FRAMES = 2084952

print("Project root:", PROJECT_ROOT)
print("Source frames:", SOURCE_FRAMES_DIR)
print("Output dataset:", OUTPUT_DIR)

Project root: /home/hemanth/Desktop/CHOLEC80-CVS-PUBLIC
Source frames: /home/hemanth/Desktop/CHOLEC80-CVS-PUBLIC/data/cholec80_frames
Output dataset: /home/hemanth/Desktop/CHOLEC80-CVS-PUBLIC/data/cholec80_cvs


## 2. Dry Run Summary

This validates the CSV references and prints the exact amount of data that will be copied. The notebook stops here if any referenced frame is missing.

In [3]:
def frame_name(image_value):
    return f"{int(image_value)}.jpg"


def source_frame_path(video_name, image_value):
    return SOURCE_FRAMES_DIR / str(video_name) / frame_name(image_value)


def split_relative_path(video_name, image_value):
    return f"{video_name}/{frame_name(image_value)}"


def root_relative_path(split_name, video_name, image_value):
    return f"{split_name}/{video_name}/{frame_name(image_value)}"


def validate_duplicate_labels(df, split_name):
    label_variants = df.groupby(["video_name", "image"])[LABEL_COLUMNS].nunique().max(axis=1)
    inconsistent = label_variants[label_variants > 1]
    if len(inconsistent) > 0:
        examples = list(inconsistent.index[:5])
        raise ValueError(f"{split_name} has duplicate frame rows with conflicting labels: {examples}")


split_rows = {}
split_unique_rows = {}
summary = {}
all_unique_source_paths = set()

for split_name, csv_path in SPLIT_CSVS.items():
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing split CSV: {csv_path}")

    df = pd.read_csv(csv_path)
    expected_columns = {"video_name", "image", *LABEL_COLUMNS}
    missing_columns = expected_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f"{csv_path} is missing columns: {sorted(missing_columns)}")

    validate_duplicate_labels(df, split_name)

    df = df.copy()
    df["relative_path"] = [split_relative_path(r.video_name, r.image) for r in df.itertuples(index=False)]
    split_rows[split_name] = df

    unique_df = df.drop_duplicates(subset=["video_name", "image"]).copy()
    unique_df["source_path"] = [source_frame_path(r.video_name, r.image) for r in unique_df.itertuples(index=False)]
    unique_df["root_relative_path"] = [root_relative_path(split_name, r.video_name, r.image) for r in unique_df.itertuples(index=False)]
    split_unique_rows[split_name] = unique_df

    missing = [path for path in unique_df["source_path"] if not path.exists()]
    if missing:
        examples = "\n".join(str(path) for path in missing[:10])
        raise FileNotFoundError(f"{split_name} has {len(missing)} missing source frames. Examples:\n{examples}")

    size_bytes = sum(path.stat().st_size for path in unique_df["source_path"])
    all_unique_source_paths.update(unique_df["source_path"])
    summary[split_name] = {
        "rows": int(len(df)),
        "unique_frames": int(len(unique_df)),
        "duplicate_rows": int(len(df) - len(unique_df)),
        "videos": int(df["video_name"].nunique()),
        "size_bytes": int(size_bytes),
        "size_gib": round(size_bytes / 1024**3, 3),
    }

total_unique_frames = sum(item["unique_frames"] for item in summary.values())
total_size_bytes = sum(item["size_bytes"] for item in summary.values())

print(json.dumps(summary, indent=2))
print("Total unique frames to copy:", total_unique_frames)
print("Approx copy size GiB:", round(total_size_bytes / 1024**3, 3))

if total_unique_frames != EXPECTED_TOTAL_UNIQUE_FRAMES:
    raise AssertionError(f"Expected {EXPECTED_TOTAL_UNIQUE_FRAMES} unique frames, found {total_unique_frames}")

print("Dry run passed. No missing frame references found.")

{
  "train": {
    "rows": 43850,
    "unique_frames": 43842,
    "duplicate_rows": 8,
    "videos": 50,
    "size_bytes": 5793597481,
    "size_gib": 5.396
  },
  "valid": {
    "rows": 9615,
    "unique_frames": 9615,
    "duplicate_rows": 0,
    "videos": 15,
    "size_bytes": 1081261316,
    "size_gib": 1.007
  },
  "test": {
    "rows": 9295,
    "unique_frames": 9295,
    "duplicate_rows": 0,
    "videos": 15,
    "size_bytes": 1047447070,
    "size_gib": 0.976
  }
}
Total unique frames to copy: 62752
Approx copy size GiB: 7.378
Dry run passed. No missing frame references found.


## 3. Copy Frames and Write Manifests

This cell creates `data/cholec80_csv_uploadable_5fps/`. Existing files with the same size are skipped so the cell can be safely rerun.

In [4]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def copy_frame(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        return "skipped"
    shutil.copy2(src, dst)
    return "copied"


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "annotations").mkdir(parents=True, exist_ok=True)

manifest_rows = []
copy_counts = {split_name: {"copied": 0, "skipped": 0} for split_name in SPLIT_CSVS}

for split_name, df in split_rows.items():
    split_dir = OUTPUT_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    split_annotations = df.copy()
    split_annotations.to_csv(split_dir / "annotations.csv", index=False)

    unique_df = split_unique_rows[split_name]
    split_manifest_rows = []

    iterator = unique_df.itertuples(index=False)
    for row in tqdm(iterator, total=len(unique_df), desc=f"Copying {split_name}"):
        src = Path(row.source_path)
        dst = OUTPUT_DIR / row.root_relative_path
        status = copy_frame(src, dst)
        copy_counts[split_name][status] += 1

        file_size = dst.stat().st_size
        checksum = sha256_file(dst)
        manifest_row = {
            "split": split_name,
            "video_name": row.video_name,
            "image": int(row.image),
            "relative_path": row.root_relative_path,
            "source_path": str(src),
            "two_structures_score": int(row.two_structures_score),
            "cystic_plate_score": int(row.cystic_plate_score),
            "hc_triangle_score": int(row.hc_triangle_score),
            "file_size_bytes": int(file_size),
            "sha256": checksum,
        }
        manifest_rows.append(manifest_row)
        split_manifest_rows.append(manifest_row)

    pd.DataFrame(split_manifest_rows).to_csv(split_dir / "manifest.csv", index=False)

root_manifest = pd.DataFrame(manifest_rows)
root_manifest.to_csv(OUTPUT_DIR / "manifest.csv", index=False)
root_manifest[["relative_path", "file_size_bytes", "sha256"]].to_csv(OUTPUT_DIR / "checksums_sha256.csv", index=False)

print(json.dumps(copy_counts, indent=2))
print("Wrote root manifest rows:", len(root_manifest))

Copying train:   0%|          | 0/43842 [00:00<?, ?it/s]

Copying valid:   0%|          | 0/9615 [00:00<?, ?it/s]

Copying test:   0%|          | 0/9295 [00:00<?, ?it/s]

{
  "train": {
    "copied": 43842,
    "skipped": 0
  },
  "valid": {
    "copied": 9615,
    "skipped": 0
  },
  "test": {
    "copied": 9295,
    "skipped": 0
  }
}
Wrote root manifest rows: 62752


## 4. Copy Annotation Assets and Write Documentation

This keeps the uploadable dataset self-describing.

In [5]:
annotations_dir = OUTPUT_DIR / "annotations"

shutil.copy2(SPLIT_CSVS["train"], annotations_dir / "train.csv")
shutil.copy2(SPLIT_CSVS["valid"], annotations_dir / "valid.csv")
shutil.copy2(SPLIT_CSVS["test"], annotations_dir / "test.csv")

for output_name, source_path in ANNOTATION_ASSETS.items():
    if source_path.exists():
        shutil.copy2(source_path, annotations_dir / output_name)

metadata = {
    "dataset_name": "cholec80_csv_uploadable_5fps",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root_at_creation": str(PROJECT_ROOT),
    "source_frames_dir": str(SOURCE_FRAMES_DIR),
    "output_dir": str(OUTPUT_DIR),
    "original_video_fps": ORIGINAL_VIDEO_FPS,
    "target_csv_fps": TARGET_FPS,
    "full_extracted_frame_count_reference": EXPECTED_FULL_EXTRACTED_FRAMES,
    "exported_unique_frame_count": int(len(root_manifest)),
    "exported_row_count": int(sum(len(df) for df in split_rows.values())),
    "split_summary": summary,
    "split_name_mapping": {
        "train": "data/train.csv",
        "valid": "data/val.csv",
        "test": "data/test.csv",
    },
    "label_columns": LABEL_COLUMNS,
    "label_note": "Training code binarizes scores with min(1, score), so 2 is treated as positive.",
}

with open(OUTPUT_DIR / "metadata.json", "w") as handle:
    json.dump(metadata, handle, indent=2)

readme = f"""# Cholec80-CSV Uploadable 5 FPS Subset

This folder contains only the frames referenced by the Cholec80-CSV train, validation, and test CSV files.

It is intended to be uploaded or moved without the full `data/cholec80_frames` directory.

## Structure

- `train/`, `valid/`, `test/`: split folders containing video subfolders and copied `.jpg` frames.
- `train/annotations.csv`, `valid/annotations.csv`, `test/annotations.csv`: split-local labels with `relative_path` values relative to the split folder.
- `manifest.csv`: file-level manifest for every copied unique frame.
- `checksums_sha256.csv`: SHA256 checksums for copied files.
- `annotations/`: copies of the source CSV/config files used to create this subset.
- `metadata.json`: machine-readable creation metadata.
- `preprocessing_summary.md`: human-readable preprocessing notes.

## Counts

- Train rows: {summary['train']['rows']:,}; unique frames: {summary['train']['unique_frames']:,}; videos: {summary['train']['videos']}
- Valid rows: {summary['valid']['rows']:,}; unique frames: {summary['valid']['unique_frames']:,}; videos: {summary['valid']['videos']}
- Test rows: {summary['test']['rows']:,}; unique frames: {summary['test']['unique_frames']:,}; videos: {summary['test']['videos']}
- Total unique copied frames: {len(root_manifest):,}

## Loading Paths

Use each annotation row's `relative_path` inside its split folder. For example, a row in `train/annotations.csv` with `relative_path = video03/94461.jpg` points to `train/video03/94461.jpg`.
"""

preprocessing_summary = f"""# Preprocessing Summary

## Source Videos

- The Cholec80 source videos are 25 FPS.
- `video_2_frames.py` extracts frames before the `ClippingCutting` phase using `results/videos_frames_index.csv`.
- The full valid extracted frame set has {EXPECTED_FULL_EXTRACTED_FRAMES:,} frames.

## Label Generation

- `annotations_2_labels.py` uses `truncation_ratio = 0.85`, keeping the final 15% region after truncation.
- It sets `video_fps = 25` and `target_fps = 5`.
- For each 25-frame second, it randomly selects 5 frames, so the CSV dataset is effectively 5 FPS.

## Labels

- CSV label columns are `two_structures_score`, `cystic_plate_score`, and `hc_triangle_score`.
- Raw CSV labels can include `0`, `1`, or `2`.
- `Cholec80CSVDataset` binarizes labels at training time with `min(1, score)`, so both `1` and `2` are treated as positive.

## Split Policy

- The split is by video, not by individual frames.
- Train: {summary['train']['videos']} videos.
- Valid: {summary['valid']['videos']} videos. This folder corresponds to repository `data/val.csv`.
- Test: {summary['test']['videos']} videos.

## Export Policy

- This dataset copies only frames referenced by `train.csv`, `val.csv`, and `test.csv`.
- It does not include the full extracted frame directory.
- Duplicate CSV rows are preserved in split annotation CSVs, but image files are copied only once per split.
"""

(OUTPUT_DIR / "README.md").write_text(readme)
(OUTPUT_DIR / "preprocessing_summary.md").write_text(preprocessing_summary)

print("Documentation and annotation assets written to:", OUTPUT_DIR)

Documentation and annotation assets written to: /home/hemanth/Desktop/CHOLEC80-CVS-PUBLIC/data/cholec80_cvs


## 5. Verification

This confirms the exported folder is complete and did not accidentally copy the full 2M-frame directory.

In [6]:
def count_jpgs(path):
    return sum(1 for _ in path.rglob("*.jpg"))


copied_jpg_count = count_jpgs(OUTPUT_DIR)
if copied_jpg_count != EXPECTED_TOTAL_UNIQUE_FRAMES:
    raise AssertionError(f"Expected {EXPECTED_TOTAL_UNIQUE_FRAMES} copied jpg files, found {copied_jpg_count}")

if copied_jpg_count >= EXPECTED_FULL_EXTRACTED_FRAMES:
    raise AssertionError("The output appears to contain the full extracted frame directory, which should not happen.")

for split_name, df in split_rows.items():
    split_dir = OUTPUT_DIR / split_name
    missing_exported = [split_dir / path for path in df["relative_path"] if not (split_dir / path).exists()]
    if missing_exported:
        examples = "\n".join(str(path) for path in missing_exported[:10])
        raise FileNotFoundError(f"{split_name} has {len(missing_exported)} missing exported frames. Examples:\n{examples}")

manifest = pd.read_csv(OUTPUT_DIR / "manifest.csv")
if len(manifest) != EXPECTED_TOTAL_UNIQUE_FRAMES:
    raise AssertionError(f"Manifest has {len(manifest)} rows, expected {EXPECTED_TOTAL_UNIQUE_FRAMES}")

print("Verification passed.")
print("Exported dataset:", OUTPUT_DIR)
print("Copied jpg files:", copied_jpg_count)
print("Manifest rows:", len(manifest))

Verification passed.
Exported dataset: /home/hemanth/Desktop/CHOLEC80-CVS-PUBLIC/data/cholec80_cvs
Copied jpg files: 62752
Manifest rows: 62752
